In [ ]:
import ee
import os
import time
import geopandas as gpd
import numpy as np
from shapely.geometry import box, mapping
from tqdm import tqdm
from pathlib import Path

## 1. 全局与藏东南区域配置

In [ ]:

GEE_PROJECT = os.getenv("GEE_PROJECT", "") # gee项目路径
REGIONS_DIR = Path("data/boundary/setp")

TARGET_REGIONS_MAP = {
    "SETP_Boundary.shp": {
        "alias": "SETP",
        "crs": "EPSG:32646",
        "melt_start": 9, "melt_end": 10,
        "cloud_max": 80,
    }
}

EXPORT_SCALE = 30
TILE_SIZE_M = 100000
BUFFER_M = 10000
DRIVE_FOLDER = "SETP_rsimg"  # Drive 文件夹名称
DRIVE_FORMAT = "GeoTIFF"

# 将 1985—2025 限制为总数据时间范围
TARGET_PERIODS = [
    ("1985", 1984, 1988),
    ("1990", 1988, 1992),
    ("1995", 1993, 1997),
    ("2000", 1998, 2002),
    ("2005", 2003, 2007),
    ("2010", 2008, 2012),
    ("2015", 2013, 2017),
    ("2020", 2019, 2021),
    ("2025", 2022, 2025),
]

OPTICAL_BANDS = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]

# 完整的多源卫星集合（包含 Tier 1 与 Tier 2 兜底）
COLLECTIONS = {
    # --- Tier 1 数据源 ---
    "L5_L2":    {"id": "LANDSAT/LT05/C02/T1_L2", "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 1984, "end_year": 2012},
    "L7_L2":    {"id": "LANDSAT/LE07/C02/T1_L2", "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 1999, "end_year": 2024},
    "L8_L2":    {"id": "LANDSAT/LC08/C02/T1_L2", "bands": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 2013, "end_year": 9999},
    "L9_L2":    {"id": "LANDSAT/LC09/C02/T1_L2", "bands": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 2021, "end_year": 9999},
    "S2_L2":    {"id": "COPERNICUS/S2_SR_HARMONIZED", "bands": ["B2", "B3", "B4", "B8", "B11", "B12"], "scale": 0.0001, "offset": 0.0, "type": "s2", "level": "L2", "start_year": 2017, "end_year": 9999},
    "L4_L1":    {"id": "LANDSAT/LT04/C02/T1_TOA", "bands": ["B1", "B2", "B3", "B4", "B5", "B7"], "scale": 1.0, "offset": 0.0, "type": "landsat", "level": "L1", "start_year": 1982, "end_year": 1993},
    "L5_L1":    {"id": "LANDSAT/LT05/C02/T1_TOA", "bands": ["B1", "B2", "B3", "B4", "B5", "B7"], "scale": 1.0, "offset": 0.0, "type": "landsat", "level": "L1", "start_year": 1984, "end_year": 2012},

    # --- Tier 2 兜底数据源 ---
    "L5_L2_T2": {"id": "LANDSAT/LT05/C02/T2_L2",  "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 1984, "end_year": 2012},
    "L7_L2_T2": {"id": "LANDSAT/LE07/C02/T2_L2",  "bands": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"], "scale": 0.0000275, "offset": -0.2, "type": "landsat", "level": "L2", "start_year": 1999, "end_year": 2024},
    "L5_L1_T2": {"id": "LANDSAT/LT05/C02/T2_TOA", "bands": ["B1", "B2", "B3", "B4", "B5", "B7"], "scale": 1.0, "offset": 0.0, "type": "landsat", "level": "L1", "start_year": 1984, "end_year": 2012},
    "L4_L1_T2": {"id": "LANDSAT/LT04/C02/T2_TOA", "bands": ["B1", "B2", "B3", "B4", "B5", "B7"], "scale": 1.0, "offset": 0.0, "type": "landsat", "level": "L1", "start_year": 1982, "end_year": 1993}
}

## 2. 网格切片生成

In [ ]:
def generate_dynamic_tiles(boundary_gdf, target_crs, tile_size_m, buffer_m):
    if boundary_gdf.empty or boundary_gdf.crs is None:
        raise ValueError("边界为空或缺少坐标参考系。")
    if not boundary_gdf.geometry.is_valid.all():
        raise ValueError("研究区含无效几何，请先修复边界。")
    proj_gdf = boundary_gdf.to_crs(target_crs)
    minx, miny, maxx, maxy = proj_gdf.total_bounds

    polygons = []
    for x in np.arange(minx, maxx, tile_size_m):
        for y in np.arange(miny, maxy, tile_size_m):
            b = box(x - buffer_m, y - buffer_m, x + tile_size_m + buffer_m, y + tile_size_m + buffer_m)
            polygons.append(b)

    grid_gdf = gpd.GeoDataFrame({'geometry': polygons}, crs=target_crs)
    try:
        unified_boundary = proj_gdf.union_all()
    except AttributeError:
        unified_boundary = proj_gdf.unary_union

    intersecting_grid = grid_gdf[grid_gdf.intersects(unified_boundary)].copy()
    return intersecting_grid.to_crs("EPSG:4326")

## 3. 影像合成

In [ ]:
def harmonize_to_l8(img, sensor):
    ROY2016 = {
        "L4": {"Blue": (0.8850, 0.0183), "Green": (0.9317, 0.0123), "Red": (0.9372, 0.0123), "NIR": (0.8339, 0.0448), "SWIR1": (0.8639, 0.0306), "SWIR2": (0.9165, 0.0116)},
        "L5": {"Blue": (0.8850, 0.0183), "Green": (0.9317, 0.0123), "Red": (0.9372, 0.0123), "NIR": (0.8339, 0.0448), "SWIR1": (0.8639, 0.0306), "SWIR2": (0.9165, 0.0116)},
        "L7": {"Blue": (0.8474, 0.0003), "Green": (0.8483, 0.0054), "Red": (0.9047, 0.0042), "NIR": (0.8462, 0.0336), "SWIR1": (0.8937, 0.0065), "SWIR2": (0.9071, 0.0043)},
        "S2": {"Blue": (0.9778, 0.0053), "Green": (1.0053, -0.0037), "Red": (0.9765, 0.0058), "NIR": (0.9983, 0.0019), "SWIR1": (0.9987, -0.0002), "SWIR2": (1.0133, -0.0140)}
    }
    sKey = sensor.split("_")[0]
    coeffs = ROY2016.get(sKey)
    if coeffs is None: 
        return img

    bands = [
        img.select(b).multiply(coeffs[b][0]).add(coeffs[b][1]).rename(b)
        for b in OPTICAL_BANDS
    ]
    return ee.Image(ee.Image.cat(bands).copyProperties(img, ["system:time_start"]))

def preprocess_image(img, sensor):
    cfg = COLLECTIONS[sensor]

    # 辐射校正与地表反射率线性转换
    scaled = (img.select(cfg["bands"])
              .multiply(cfg["scale"])
              .add(cfg.get("offset", 0.0))
              .clamp(0.0, 1.0)
              .rename(OPTICAL_BANDS))

    # 严格黑边与填充掩膜：确保所选的 6 个波段都不是 0（剔除切片黑边及 L7 SLC-off 扫描带空隙）
    valid_footprint = img.select(cfg["bands"]).reduce(ee.Reducer.min()).gt(0)
    scaled = scaled.updateMask(valid_footprint)

    if cfg.get("level") == "L1":
        scaled = scaled.multiply(0.85)

    return harmonize_to_l8(scaled.toFloat(), sensor)

def build_period_composite(start_year, end_year, region_geom, melt_start, melt_end, cloud_thresh):
    collections_to_merge = []

    for sname, cfg in COLLECTIONS.items():
        if cfg["start_year"] > end_year or cfg["end_year"] < start_year:
            continue

        cloud_prop = "CLOUDY_PIXEL_PERCENTAGE" if cfg["type"] == "s2" else "CLOUD_COVER"

        col = (ee.ImageCollection(cfg["id"])
               .filterBounds(region_geom)
               .filter(ee.Filter.calendarRange(start_year, end_year, "year"))
               .filter(ee.Filter.calendarRange(melt_start, melt_end, "month"))
               .filter(ee.Filter.Or(
                   ee.Filter.lt(cloud_prop, cloud_thresh),
                   ee.Filter.eq(cloud_prop, None)
               )))

        processed_col = col.map(lambda img, s=sname: preprocess_image(img, s))
        collections_to_merge.append(processed_col)

    # 规范集合合并：避免使用 unbounded 的 dummy_empty 常数图
    if not collections_to_merge:
        raise ValueError(f"在 {start_year}-{end_year} 期间未找到任何匹配的卫星数据配置！")

    merged_col = collections_to_merge[0]
    for c in collections_to_merge[1:]:
        merged_col = merged_col.merge(c)

    # 纯中值合成（最大限度避免像元被抠空）
    optical_mosaic = merged_col.median()

    # 关键修复：加入 clamp(0, 10000)，杜绝阴影/负截距导致的小于 0 反射率在转 uint16 时发生下溢（65535白斑）
    optical_uint16 = (optical_mosaic.multiply(10000)
                      .round()
                      .clamp(0, 10000)
                      .toUint16())

    final_output = optical_uint16.clip(region_geom)

    return final_output

## 4. 主程序：批量切片派发与进度轮询

In [ ]:
def main():
    try: 
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        print("[AUTH] 本地凭证失效或不匹配，正在请求重新授权...")
        ee.Authenticate(auth_mode='localhost', force=True)
        ee.Initialize(project=GEE_PROJECT)

    active_tasks = {}

    for shp_name, region_info in TARGET_REGIONS_MAP.items():
        region_alias = region_info["alias"]
        region_crs   = region_info["crs"]
        melt_s       = region_info["melt_start"]
        melt_e       = region_info["melt_end"]
        cloud_max    = region_info["cloud_max"]

        shp_filepath = os.path.join(REGIONS_DIR, shp_name)
        if not os.path.exists(shp_filepath):
            print(f"[WARN] 找不到边界文件: {shp_filepath}，跳过该区域。")
            continue

        print("\n" + "=" * 50)
        print(f"处理区域: {region_alias} | 投影: {region_crs} | 窗口: {melt_s}-{melt_e}月 | 云量阈值: {cloud_max}%")
        print("=" * 50)

        for suffix in (".shp", ".shx", ".dbf", ".prj"):
            if not Path(shp_filepath).with_suffix(suffix).is_file():
                raise FileNotFoundError(Path(shp_filepath).with_suffix(suffix))
        boundary_gdf = gpd.read_file(shp_filepath)
        valid_tiles  = generate_dynamic_tiles(boundary_gdf, region_crs, TILE_SIZE_M, BUFFER_M)
        print(f"[INFO] 切片构建完成，有效网格数: {len(valid_tiles)}")

        study_geom_ee = ee.Geometry(mapping(boundary_gdf.to_crs("EPSG:4326").unary_union))
        for target_name, start_yr, end_yr in TARGET_PERIODS:
            print(f"\n--- 派发年代 {target_name} ({start_yr}-{end_yr}) ---")

            for index_counter, (_, row) in tqdm(enumerate(valid_tiles.iterrows()), total=len(valid_tiles), desc=f"{region_alias}-{target_name}"):
                tile_id     = f"Tile_{index_counter+1:04d}"
                description = f"{region_alias}_{target_name}_{tile_id}_30m"

                tile_geom_ee = ee.Geometry(mapping(row.geometry)).intersection(study_geom_ee, 1)

                try:
                    composite = build_period_composite(start_yr, end_yr, tile_geom_ee, melt_s, melt_e, cloud_max)

                    # Google Drive folder name.
                    target_folder = f"{DRIVE_FOLDER}_{region_alias}_{target_name}"

                    task = ee.batch.Export.image.toDrive(
                        image=composite, 
                        description=description, 
                        folder=target_folder,
                        crsTransform=[EXPORT_SCALE, 0, 0, 0, -EXPORT_SCALE, 0], 
                        region=tile_geom_ee, 
                        crs=region_crs, 
                        maxPixels=1e13, 
                        fileFormat=DRIVE_FORMAT, 
                        formatOptions={"cloudOptimized": True}
                    )
                    task.start()
                    active_tasks[task.id] = {"description": description, "task": task}


                except Exception as e: 
                    print(f"\n[ERROR] 派发失败 {description}: {e}")

    # 轮询监听任务进度
    poll_interval = 60
    while active_tasks:
        time.sleep(poll_interval)
        completed_ids = []
        for tid, info in list(active_tasks.items()):
            try:
                state = info["task"].status().get("state")
                if state == "COMPLETED":
                    print(f"\n[DONE] {info['description']} -> Google Drive")
                    completed_ids.append(tid)
                elif state in ("FAILED", "CANCELLED"):
                    print(f"\n[FAILED] {info['description']}: {info['task'].status().get('error_message')}")
                    completed_ids.append(tid)
            except Exception:
                pass

        for tid in completed_ids: 
            del active_tasks[tid]

        if active_tasks: 
            print(f"\r[STATUS] 剩余 {len(active_tasks)} 个任务运行中...", end="", flush=True)

    print("\n\n所有研究区及年代切片的导出任务已全部执行完毕。")

## 5. 执行入口

导出到 Google Drive；

In [ ]:
main()
